### 2. Conceptos e ideas a repasar (Dominio de la Frecuencia)

* **¿Qué son las imágenes base y de qué tamaño son?**
  Son los "ladrillos" fundamentales (ondas senoidales y cosenoidales bidimensionales de distintas frecuencias y orientaciones) que, al sumarse, reconstruyen la imagen original. Si la imagen a analizar es de M x N, cada imagen base también será exactamente de M x N.

* **¿De qué tamaño es la TDF de una imagen de M x N?**
  Es exactamente de M x N. Por cada píxel en el dominio espacial, se genera un coeficiente complejo en el dominio frecuencial.

* **¿Cómo es la distribución de la energía en la TDF?**
  La energía decae fuertemente hacia los extremos. El pico máximo (los píxeles más brillantes) se concentra en el **centro** del espectro (las bajas frecuencias, que representan colores planos y fondos suaves). Hacia los **bordes** la energía es muchísimo menor (ahí viven las altas frecuencias: bordes bruscos, texturas finas y ruido).

* **¿Qué propiedades tiene la TDF?**
  * **Traslación:** Si movés un objeto en la foto espacialmente, la Magnitud (el módulo) *no cambia* absolutamente nada, solo cambia la Fase.
  * **Rotación:** Si rotás la imagen, el espectro de Magnitud rota exactamente los mismos grados y en la misma dirección.
  * **Simetría:** Al procesar imágenes reales, el espectro de magnitud es perfectamente simétrico (como un espejo) respecto a su centro.

* **¿Qué información se puede encontrar en el módulo y cuál en la fase?**
  * **Módulo (Magnitud):** Indica *"cuánto"* hay de cada frecuencia. Aporta el contraste general y la fuerza de los patrones.
  * **Fase:** Indica *"dónde"* está ubicada cada frecuencia. Contiene toda la información geométrica y la estructura (la "forma") de los objetos de la imagen.

* **¿Recuerda cuándo y cómo se manifiesta el fenómeno de Gibbs en las imágenes?**
  Se manifiesta como un efecto de anillado (*ringing* o "ondas en el agua") alrededor de los bordes nítidos de la imagen. Ocurre cuando se aplica un filtro **ideal** en el dominio frecuencial (un filtro con un corte abrupto o "pared de ladrillo", que genera estas oscilaciones al volver al dominio espacial).

In [1]:
import cv2
import numpy as np
import timeit
import matplotlib.pyplot as plt

# Creamos una imagen aleatoria grande (1024x1024) para que la diferencia de tiempo sea notoria
img = np.random.rand(1024, 1024).astype(np.float32)

print("--- BENCHMARK: Transformada Discreta de Fourier (TDF) ---")

# ==========================================
# 1. MÉTODO NUMPY (El "Amigable")
# ==========================================
# ¿Por qué es amigable? Porque toma la imagen directamente y devuelve una matriz 
# de números complejos de Python, lista para usar.
inicio_np = timeit.default_timer()
fft_numpy = np.fft.fft2(img)
fft_shift_numpy = np.fft.fftshift(fft_numpy) # Centrado
fin_np = timeit.default_timer()

tiempo_np = (fin_np - inicio_np) * 1000 # Convertimos a milisegundos

# ==========================================
# 2. MÉTODO OPENCV (El "Rápido")
# ==========================================
# ¿Por qué es menos amigable? Porque exige que la imagen sea float32 estricto, 
# requiere flags, y devuelve una matriz de 2 canales (Canal 1: Real, Canal 2: Imaginario) 
# en lugar de números complejos nativos.
inicio_cv = timeit.default_timer()
fft_opencv = cv2.dft(img, flags=cv2.DFT_COMPLEX_OUTPUT)
# El centrado en OpenCV hay que hacerlo a mano o usando la función de numpy igual
fft_shift_opencv = np.fft.fftshift(fft_opencv) 
fin_cv = timeit.default_timer()

tiempo_cv = (fin_cv - inicio_cv) * 1000 # Convertimos a milisegundos

# ==========================================
# RESULTADOS
# ==========================================
print(f"Tiempo NumPy:  {tiempo_np:.2f} ms")
print(f"Tiempo OpenCV: {tiempo_cv:.2f} ms")

if tiempo_cv < tiempo_np:
    print(f"\n🏆 Ganador: OpenCV (Es un {(tiempo_np / tiempo_cv):.1f}x más rápido)")
else:
    print(f"\n🏆 Ganador: NumPy (Es un {(tiempo_cv / tiempo_np):.1f}x más rápido)")

--- BENCHMARK: Transformada Discreta de Fourier (TDF) ---
Tiempo NumPy:  109.04 ms
Tiempo OpenCV: 26.87 ms

🏆 Ganador: OpenCV (Es un 4.1x más rápido)


### Conclusión: OpenCV vs. NumPy en la Transformada de Fourier

Tras realizar el benchmark de rendimiento, queda comprobado empíricamente que **OpenCV es superior en velocidad**, pero **NumPy gana por goleada en facilidad de uso**.

* 🏆 **El más rápido (OpenCV - `cv.dft`):**
  * **Por qué es más veloz:** Está programado en rutinas de C/C++ hiper-optimizadas específicamente para operaciones matriciales de imágenes y visión artificial.
  * **Por qué es "menos amigable":** Es muy estricto con los tipos de datos. Obliga a convertir la imagen a `np.float32`, exige el uso de *flags* (`cv2.DFT_COMPLEX_OUTPUT`), y en lugar de devolver números complejos reales, devuelve un tensor de dos canales (Canal 1 para la parte Real, Canal 2 para la Imaginaria). Esto obliga a usar funciones adicionales como `cv.magnitude()` para poder visualizar algo.

* 🤝 **El más amigable (NumPy - `np.fft.fft2`):**
  * **Por qué es amigable:** Su sintaxis es directa y transparente. Toma la imagen y devuelve automáticamente una matriz de **números complejos nativos de Python** ($a + bj$). Al usar `np.fft.fftshift`, el centrado del espectro se resuelve en una sola línea de manera muy intuitiva.
  * **Por qué es más lento:** Al ser una herramienta matemática de propósito general, tiene una ligera sobrecarga de procesamiento en Python que la hace quedar por detrás del motor especializado de OpenCV.